##### 版權所有 2024 Google LLC.


In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini 2.0 - 多模態即時 API：工具使用


<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/doggy8088/gemini-api-cookbook/blob/main/gemini-2/live_api_tool_use.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />在 Google Colab 中執行</a>
  </td>
</table>


這個筆記本提供了如何使用與 [Gemini 2.0](https://ai.google.dev/gemini-api/docs/models/gemini-v2) 的多模態即時 API 的工具範例。

這個 API 提供 Google 搜尋、程式執行和函式呼叫工具。早期的 Gemini 模型支持這些工具的版本。Gemini 2（在即時 API 中）最大的變化是，基本上所有的工具都由程式執行來處理。隨著這一變化，你可以在單一的 API 呼叫中使用 **多個工具** ，並且模型可以在單一的程式執行區塊中使用多個工具。

本教程假設你已熟悉即時 API，如 [此教程](https://github.com/google-gemini/cookbook/blob/main/gemini-2/live_api_starter.ipynb) 中所描述。


## 設定


### 安裝 SDK

新的 **[Google Gen AI SDK](https://ai.google.dev/gemini-api/docs/sdks)** 提供對 Gemini 2.0（及之前版本）的程式化訪問，使用 [Google AI for Developers](https://ai.google.dev/gemini-api/docs) 和 [Vertex AI](https://cloud.google.com/vertex-ai/generative-ai/docs/overview) 兩個 API。除幾個例外外，運行在一個平台上的程式碼可以在兩個平台上運行。這意味著你可以使用開發者 API 來原型化應用程式，然後將該應用程式遷移到 Vertex AI，而無需重寫你的程式碼。

有關這個新 SDK 的更多詳情，請參考 [文檔](https://ai.google.dev/gemini-api/docs/sdks) 或查看 [快速入門](../gemini-2/get_started.ipynb) 筆記本。


In [1]:
!pip install -U -q google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.3/110.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.2/168.2 kB 6.9 MB/s eta 0:00:00


### 設定你的 API 金鑰

要執行以下的區塊，你的 API 金鑰必須儲存在名為 `GOOGLE_API_KEY` 的 Colab 祕密中。如果你還沒有 API 金鑰，或者不確定如何建立 Colab 祕密，請參考 [驗證](../quickstarts/Authentication.ipynb) 的範例。


In [2]:
from google.colab import userdata
import os

os.environ['GOOGLE_API_KEY']=userdata.get('GOOGLE_API_KEY')

### 初始化 SDK 客戶端

客戶端將從環境變數中取得你的 API 金鑰。  
要使用即時 API，你需要將客戶端版本設置為 `v1alpha`。


In [3]:
from google import genai

client = genai.Client(http_options= {
      'api_version': 'v1alpha'
})

### 選擇模型

多模式即時 API 是與 [Gemini 2.0](https://ai.google.dev/gemini-api/docs/models/gemini-v2) 模型一起引入的新能力。它無法與先前的模型運作。


In [4]:
model_name = "gemini-2.0-flash-exp"

### 匯入


In [5]:
import asyncio
import contextlib
import json
import wave

from IPython import display

from google import genai
from google.genai import types

### 工具程式


你將使用 Live API 的音訊輸出，最簡單的方式在 Colab 中聽到它是將 `PCM` 資料寫出為 `WAV` 檔案：


In [6]:
@contextlib.contextmanager
def wave_file(filename, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        yield wf

使用紀錄器讓切換開/關除錯訊息變得更容易。


In [7]:
import logging
logger = logging.getLogger('Live')
#logger.setLevel('DEBUG')  # Switch between "INFO" and "DEBUG" to toggle debug messages.
logger.setLevel('INFO')

## 開始使用


大部分的 Live API 設定將與 [入門教學](../gemini-2/live_api_starter.ipynb) 相似。由於這個教學不專注於 API 的即時互動性，程式碼已被簡化：這段程式碼使用了 Live API，但它只發送了一個單一的文字提示，並監聽單一輪的回覆。


In [8]:
n = 0
async def run(prompt, modality='TEXT', tools=None):
  global n
  if tools is None:
    tools=[]

  config = {
          "tools": tools,
          "generation_config": {
              "response_modalities": [modality]}}

  async with client.aio.live.connect(model=model_name, config=config) as session:
    display.display(display.Markdown(prompt))
    display.display(display.Markdown('-------------------------------'))
    await session.send(prompt, end_of_turn=True)

    audio = False
    filename = f'audio_{n}.wav'
    with wave_file(filename) as wf:
      async for response in session.receive():
        logger.debug(str(response))

        server_content = response.server_content
        if server_content is not None:
          a = handle_server_content(wf, server_content)
          audio = audio or a

        tool_call = response.tool_call
        if tool_call is not None:
          await handle_tool_call(session, tool_call)


  if audio:
    display.display(display.Audio(filename, autoplay=True))
    n = n+1

由於這個教程展示了幾個工具，你需要更多程式碼來處理它所返回的不同類型的物件。

- `code_execution` 工具可以返回 `executable_code` 和 `code_execution_result` 部分。
- `google_search` 工具可能會附加一個 `grounding_metadata` 物件。


In [11]:
def handle_server_content(wf, server_content):
  audio = False
  model_turn = server_content.model_turn
  if model_turn:
    for part in model_turn.parts:
      text = part.text
      if text is not None:
        display.display(display.Markdown(text))

      inline_data = part.inline_data
      if inline_data is not None:
        print('.', end='')
        pcm_data = inline_data.data
        wf.writeframes(pcm_data)
        audio = True

      executable_code = part.executable_code
      if executable_code is not None:
        display.display(display.Markdown('-------------------------------'))
        display.display(display.Markdown(f'``` python\n{executable_code.code}\n```'))
        display.display(display.Markdown('-------------------------------'))

      code_execution_result = part.code_execution_result
      if code_execution_result is not None:
        display.display(display.Markdown('-------------------------------'))
        display.display(display.Markdown(f'```\n{code_execution_result.output}\n```'))
        display.display(display.Markdown('-------------------------------'))

  grounding_metadata = getattr(server_content, 'grounding_metadata', None)
  if grounding_metadata is not None:
    display.display(
        display.HTML(grounding_metadata.search_entry_point.rendered_content))

  return audio

- 最終，使用 `function_declarations` 工具，API 可能會回傳 `tool_call` 物件。為了保持這段程式碼的簡潔，`tool_call` 處理器僅對每個函式呼叫回覆一個 `"ok"` 的回應。


In [10]:
async def handle_tool_call(session, tool_call):
  for fc in tool_call.function_calls:
    tool_response = types.LiveClientToolResponse(
        function_responses=[types.FunctionResponse(
            name=fc.name,
            id=fc.id,
            response={'result':'ok'},
        )]
    )

    print('>>> ', tool_response)
    await session.send(tool_response)

嘗試第一次執行它：


In [ ]:
await run(prompt="Hello?", tools=None, modality = "TEXT")

Hello?

-------------------------------

Hello

 there! How can I help you today?


## 簡單的函式呼叫


API 的函式呼叫功能可以處理各種函式。SDK 中的支援仍在建構中。因此保持簡單，只需發送一個最小的函式定義：僅包含函式的名稱。

請注意，在即時 API 中，函式呼叫與聊天輪次是獨立的。在函式呼叫正在處理時，對話仍然可以繼續進行。


In [ ]:
turn_on_the_lights = {'name': 'turn_on_the_lights'}
turn_off_the_lights = {'name': 'turn_off_the_lights'}

In [ ]:
prompt = "Turn on the lights"

tools = [
    {'function_declarations': [turn_on_the_lights, turn_off_the_lights]}
]

await run(prompt, tools=tools, modality = "TEXT")

Turn on the lights

-------------------------------

-------------------------------

``` python
print(default_api.turn_on_the_lights())

```

-------------------------------

>>>  function_responses=[FunctionResponse(id='function-call-9277519295107782492', name='turn_on_the_lights', response={'result': 'ok'})]


-------------------------------

```
{'result': 'ok'}

```

-------------------------------

OK

## 程式碼執行


The `code_execution` 讓模型撰寫並執行 python 程式碼。嘗試在一個模型無法從記憶中解決的數學問題上：


In [ ]:
prompt="What is the largest prime palindrome under 100000."

tools = [
    {'code_execution': {}}
]

await run(prompt, tools=tools, modality='TEXT')

What is the largest prime palindrome under 100000.

-------------------------------

Okay

, I understand. You're asking for the largest prime number that is also

 a palindrome (reads the same forwards and backward) and is less than 1

...

Okay

, I have found the largest prime palindrome under 100,00

0.

The code generated a list of palindromes by checking every number from

 99999 downwards. It then tested each of the numbers to see if it was prime. The first prime palindrome found going downwards was 9

8689, so that is the largest one.

Therefore, the answer is 98689.


## 組合函式呼叫

組合函式呼叫是指將使用者定義的函式與 `code_execution` 工具結合的能力。模型將把它們寫入更大的程式碼區塊中，然後在等待你對每個呼叫發回響應時暫停執行。


In [ ]:
prompt="Can you turn on the lights wait 10s and then turn them off?"

tools = [
    {'code_execution': {}},
    {'function_declarations': [turn_on_the_lights, turn_off_the_lights]}
]

await run(prompt, tools=tools, modality='TEXT')

Can you turn on the lights wait 10s and then turn them off?

-------------------------------

-------------------------------

``` python
import time
default_api.turn_on_the_lights()
time.sleep(10)
default_api.turn_off_the_lights()

```

-------------------------------

>>>  function_responses=[FunctionResponse(id='function-call-6184362039141552989', name='turn_on_the_lights', response={'result': 'ok'})]
>>>  function_responses=[FunctionResponse(id='function-call-17968678311537051', name='turn_off_the_lights', response={'result': 'ok'})]


## Google 搜尋


`google_search` 工具允許模型進行 google 搜尋。例如，可以嘗試詢問最近的事件，這些事件過於接近而無法包含在訓練資料中。

搜尋仍然在 `AUDIO` 模式下執行，但你無法看到詳細結果。因此，請切換到文字模式以查看完整輸出：


In [ ]:
prompt="Can you use google search tell me about the largest earthquake in california the week of Dec 5 2024?"

tools = [
   {'google_search': {}}
]

await run(prompt, tools=tools, modality='TEXT')

Can you use google search tell me about the largest earthquake in california the week of Dec 5 2024?

-------------------------------

-------------------------------

``` python
print(google_search.search(queries=["largest earthquake in California week of December 5 2024", "California earthquakes week of December 5 2024"]))

```

-------------------------------

...

Magnitude:** The earthquake was measured at a magnitude of 7.0, making it a major seismic event.
*   **Location:** It was centered offshore, about 60 miles west of Ferndale in Humboldt County, Northern California

. This region is where three tectonic plates meet, making it one of the most seismically active areas in California.
*   **Tsunami Warning:** A tsunami warning was issued for the coast of Northern California and Southern Oregon following the quake

. The warning extended from Davenport, California, to south of Florence, Oregon and included more than 4.6 million people. People were urged to move to higher ground, and Oregon State Parks closed access to its state park beaches.
*   **Tsunami Cancelled:** Fortunately, the tsunami warning was cancelled within

 a couple hours as no significant waves were reported.
*   **Aftershocks:** There were numerous aftershocks, including several that were magnitude 4.0 and greater. The strongest aftershock was a magnitude 4.7.
*   **Impact:** The earthquake caused some shaking as far south as

 the Bay Area. There were reports of minor damage including broken windows, ruptured water pipes, and items knocked off store shelves.

This earthquake was the strongest in the region since 2005 when a magnitude 7.2 earthquake occurred.


## 多工具


新的 API 最大的不同之處在於，你不再局限於每個請求使用 1 個工具。嘗試結合前面幾個部分的任務：


In [ ]:
prompt = """\
  Hey, I need you to do three things for me.

  1. Then compute the largest prime plaindrome under 100000.
  2. Then use google search to lookup unformation about the largest earthquake in california the week of Dec 5 2024?
  3. Turn on the lights

  Thanks!
  """

tools = [
    {'google_search': {}},
    {'code_execution': {}},
    {'function_declarations': [turn_on_the_lights, turn_off_the_lights]}
]

await run(prompt, tools=tools, modality="TEXT")

  Hey, I need you to do three things for me.

  1. Then compute the largest prime plaindrome under 100000.
  2. Then use google search to lookup unformation about the largest earthquake in california the week of Dec 5 2024?
  3. Turn on the lights

  Thanks!
  

-------------------------------

Okay

, I will perform those tasks for you. First, let's find the

 largest prime palindrome under 100000.


...

.


-------------------------------

``` python
default_api.turn_on_the_lights()

```

-------------------------------

>>>  function_responses=[FunctionResponse(id='function-call-16138398244891385862', name='turn_on_the_lights', response={'result': 'ok'})]


## 下一步

- 有關 SDK 的更多資訊，請參閱 [SDK 文件](https://googleapis.github.io/python-genai/) 
- 此教學使用高階 SDK，如果你對較低階的細節感興趣，請嘗試 [此教學的 Websocket 版本](../gemini-2/websocket/search_tool.ipynb) 
- 此教學僅涵蓋這些工具的 _基本_ 使用，欲瞭解更深入（且更有趣）的範例，請參閱 [搜尋工具教學](../gemini-2/search_tool.ipynb)

或查看來自 [Cookbook](https://github.com/google-gemini/cookbook/blob/main/gemini-2/) 的其他 Gemini 2.0 功能，特別是這個其他 [多工具](../gemini-2/plotting_and_mapping.ipynb) 範例和關於 Gemini [空間能力](../gemini-2/spatial_understanding.ipynb) 的範例。
